# Threading vs Multiprocessing vs asyncio - Which Concurrency Tool When?

Difficulty: Advanced | ~45 min | Requires Lab 5 (recommended)

Same task, four strategies, two tables. CPU-bound is won by processes;
I/O-bound is crushed by asyncio.

In [1]:
!pip install tabulate==0.10.0


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\Dell\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


## Section 0 - The two workloads

CPU-bound: `blur(image)` - pure Python box blur, never releases the GIL.
I/O-bound: `fetch(n)` - fake 50 ms network round trip.

In [2]:
import time
from workloads import blur, fetch, IMAGES

# timeit wraps any callable and returns (result, elapsed_seconds)
def timeit(fn, *args):
    """Run fn(*args), return (result, elapsed_seconds)."""
    start = time.perf_counter()
    result = fn(*args)
    return result, time.perf_counter() - start

# Sanity check: blur one image to verify the workload works
blur(IMAGES[0])
print(f"Task: blur {len(IMAGES)} images of {len(IMAGES[0])}x{len(IMAGES[0])}")

Task: blur 100 images of 100x100


## Section 1 - Part 1: the sequential baseline (CPU)

Blur 100 images one at a time. This is the number every other strategy must beat.

In [3]:
# Sequential: blur all 100 images in a list comprehension - the baseline to beat
# _ discards the result list, cpu_seq holds only the elapsed time
_, cpu_seq = timeit(lambda: [blur(img) for img in IMAGES])
print(f"sequential: 100 images in {cpu_seq:.2f}s")

sequential: 100 images in 2.20s


## Section 2 - Part 2: threads (CPU)

ThreadPoolExecutor spreads work across threads. The GIL blocks true
parallelism - expect no speedup over sequential.

In [4]:
from concurrent.futures import ThreadPoolExecutor

# run_threads: distribute work_fn across items using a thread pool
def run_threads(work_fn, items):
    """Distribute work_fn across items using a thread pool."""
    with ThreadPoolExecutor() as pool:
        return list(pool.map(work_fn, items))

# Threads on CPU: GIL allows only one thread to compute at a time
_, cpu_threads = timeit(lambda: run_threads(blur, IMAGES))
print(f"threads: 100 images in {cpu_threads:.2f}s")

threads: 100 images in 3.34s


## Section 3 - Part 2 (cont.): asyncio (CPU)

The coroutine never awaits anything, so the event loop has nothing to
switch to - expect no speedup either.

In [5]:
import asyncio

# process_images: wraps the same list comprehension in an async def
# But it never awaits, so the event loop never gets the wheel
async def process_images():
    return [blur(img) for img in IMAGES]

# asyncio on CPU: coroutine never awaits, so event loop never gets control.
# Result: runs start-to-finish like sequential, no speedup.
start = time.perf_counter()
await process_images()
cpu_async = time.perf_counter() - start
print(f"asyncio: 100 images in {cpu_async:.2f}s")

asyncio: 100 images in 2.67s


## Section 4 - Part 3: processes (CPU)

ProcessPoolExecutor starts fresh interpreters, each with its own GIL.
Expect the first real speedup.

In [6]:
from concurrent.futures import ProcessPoolExecutor

# run_processes: distribute work_fn across items using a process pool
def run_processes(work_fn, items):
    """Distribute work_fn across items using a process pool."""
    with ProcessPoolExecutor() as pool:
        return list(pool.map(work_fn, items))

# Processes on CPU: each process has its own GIL, real parallelism
_, cpu_proc = timeit(lambda: run_processes(blur, IMAGES))
print(f"processes: 100 images in {cpu_proc:.2f}s")

processes: 100 images in 1.96s


## Section 5 - Part 4: repeat with an I/O-bound task

Same four strategies on `fetch(n)` (50 ms sleep). The winner is whoever
waits best - not whoever computes fastest.

In [7]:
N = 100

# Sequential I/O: 100 x 50ms = ~5s, CPU sits idle the whole time
_, io_seq_elapsed = timeit(lambda: [fetch(i) for i in range(N)])
print(f"sequential: {N} requests in {io_seq_elapsed:.2f}s")

# Threads on I/O: sleep() releases the GIL, so threads wait in parallel
_, io_threads_elapsed = timeit(lambda: run_threads(fetch, range(N)))
print(f"threads: {N} requests in {io_threads_elapsed:.2f}s")

# asyncio on I/O: each coroutine awaits its sleep, loop switches between all 100
# asyncio on I/O: need async def for the event loop.
# Processes use fetch from workloads.py (can't pickle inline functions).
async def fetch_async(n):
    await asyncio.sleep(0.05)  # same 50 ms, but the loop switches tasks
    return n * 2

# Use await directly (Jupyter already has a running event loop)
start = time.perf_counter()
await asyncio.gather(*[fetch_async(i) for i in range(N)])
io_async_elapsed = time.perf_counter() - start
print(f"asyncio: {N} requests in {io_async_elapsed:.2f}s")

# Processes on I/O: spawn overhead + no GIL benefit = slowest
_, io_proc_elapsed = timeit(lambda: run_processes(fetch, range(N)))
print(f"processes: {N} requests in {io_proc_elapsed:.2f}s")

sequential: 100 requests in 5.11s
threads: 100 requests in 0.67s
asyncio: 100 requests in 0.07s
processes: 100 requests in 2.67s


## Section 6 - Reading the two ledgers

Two inverted tables: CPU wins with processes, I/O wins with asyncio.

In [8]:
from tabulate import tabulate

# print_ledger: render a comparison table - each strategy vs sequential baseline
def print_ledger(title, seq_time, times_dict):
    """Print a comparison table: each strategy vs the sequential baseline."""
    print(title)
    rows = [["sequential", f"{seq_time:.2f}s", "1x"]]
    for name, t in times_dict.items():
        speedup = seq_time / max(t, 1e-9)  # avoid division by zero
        rows.append([name, f"{t:.2f}s", f"{speedup:.2f}x"])
    print(tabulate(rows, headers=["strategy", "elapsed", "vs baseline"], tablefmt="grid"))

# CPU-bound ledger: processes should win, threads and asyncio near baseline
print_ledger(
    "CPU-bound: blurring 100 images of 100x100 (lower is better)",
    cpu_seq,
    {"threads": cpu_threads, "asyncio": cpu_async, "processes": cpu_proc},
)

print()

# I/O-bound ledger: asyncio should crush everything, processes worst
print_ledger(
    "I/O-bound: 100 fake 50 ms requests (lower is better)",
    io_seq_elapsed,
    {"threads": io_threads_elapsed, "asyncio": io_async_elapsed, "processes": io_proc_elapsed},
)

CPU-bound: blurring 100 images of 100x100 (lower is better)
+------------+-----------+---------------+
| strategy   | elapsed   | vs baseline   |
+============+===========+===============+
| sequential | 2.20s     | 1x            |
+------------+-----------+---------------+
| threads    | 3.34s     | 0.66x         |
+------------+-----------+---------------+
| asyncio    | 2.67s     | 0.83x         |
+------------+-----------+---------------+
| processes  | 1.96s     | 1.13x         |
+------------+-----------+---------------+

I/O-bound: 100 fake 50 ms requests (lower is better)
+------------+-----------+---------------+
| strategy   | elapsed   | vs baseline   |
+============+===========+===============+
| sequential | 5.11s     | 1x            |
+------------+-----------+---------------+
| threads    | 0.67s     | 7.66x         |
+------------+-----------+---------------+
| asyncio    | 0.07s     | 68.39x        |
+------------+-----------+---------------+
| processes  | 2.67s     |

## Section 7 - Optional Exercise

Find the process pool startup cost: time a bare `ProcessPoolExecutor`
with one trivial task, subtract from process timings, re-read the ledgers.

The process pool's startup cost hides inside the timings. Find it.

1. Time a bare pool that runs a single trivial task to measure pure startup cost — e.g. `pool.map(abs, [1])`. (Not `lambda`: a lambda cannot be pickled across the process boundary, which is why the real work lives in `workloads.py` in the first place.)
2. Subtract that constant from `cpu_proc` and `io_proc_elapsed` and re-read the ledgers.
3. Question: with `os.cpu_count()` workers, what speedup does the CPU task approach once startup is removed? (Hint: near the core count, not beyond.)

Why does this matter in production? If a call's real work is shorter than the pool's startup cost, multiprocessing is the wrong tool — the overhead can make it *slower* than sequential.